0. Imports and configs 


In [ ]:
import os
import random
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32

IMAGES_DIR = Path("data/images/Images")
ANNOTATIONS_DIR = Path("data/annotations/Annotation")

1. Functions 

In [ ]:
def parse_annotation(annotation_path):
    """Parse XML annotation and return bounding box normalized to [0, 1]."""
    tree = ET.parse(annotation_path)
    root = tree.getroot()

    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    obj = root.find("object")
    bndbox = obj.find("bndbox")
    xmin = int(bndbox.find("xmin").text) / width
    ymin = int(bndbox.find("ymin").text) / height
    xmax = int(bndbox.find("xmax").text) / width
    ymax = int(bndbox.find("ymax").text) / height

    return [xmin, ymin, xmax, ymax]


def load_and_preprocess(image_path, label, bbox):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img, {"class_output": label, "bbox_output": bbox}


def build_dataset(paths, labels, bboxes, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, bboxes))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

#Preproceesing 

1. Downlaoding the data, donwloads only once

In [ ]:


DATA_DIR = Path("data")

if not DATA_DIR.exists():
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "jessicali9530/stanford-dogs-dataset", "--unzip", "-p", "./data"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Dataset downloaded and extracted to ./data")
    else:
        print(f"Download failed:\n{result.stderr}")
else:
    print("Dataset already exists at ./data")

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\Owner\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 227: character maps to <undefined>


Dataset downloaded and extracted to ./data


2. Breed selection and label mapping

In [5]:
SELECTED_BREEDS = [
    "n02085620-Chihuahua",
    "n02086240-Shih-Tzu",
    "n02087394-Rhodesian_ridgeback",
    "n02088364-beagle",
    "n02089973-English_foxhound",
    "n02091032-Italian_greyhound",
    "n02092339-Weimaraner",
    "n02093256-Staffordshire_bullterrier",
    "n02094433-Yorkshire_terrier",
    "n02096585-Boston_bull",
    "n02097047-miniature_schnauzer",
    "n02099601-golden_retriever",
    "n02099712-Labrador_retriever",
    "n02100877-Irish_setter",
    "n02102318-cocker_spaniel",
    "n02105855-Shetland_sheepdog",
    "n02106166-Border_collie",
    "n02106382-Bouvier_des_Flandres",
    "n02106550-Rottweiler",
    "n02106662-German_shepherd",
    "n02107142-Doberman",
    "n02107683-Bernese_mountain_dog",
    "n02108089-boxer",
    "n02108915-French_bulldog",
    "n02109047-Great_Dane",
    "n02109525-Saint_Bernard",
    "n02110185-Siberian_husky",
    "n02110958-pug",
    "n02111889-Samoyed",
    "n02112018-Pomeranian",
]

breed_to_label = {breed: i for i, breed in enumerate(SELECTED_BREEDS)} #labels are numerical class
label_to_breed = {i: breed.split("-", 1)[1] for i, breed in enumerate(SELECTED_BREEDS)}
NUM_CLASSES = len(SELECTED_BREEDS)

print(f"Selected {NUM_CLASSES} breeds:")
for i, breed in label_to_breed.items():
    print(f"  {i:2d}: {breed}")

Selected 30 breeds:
   0: Chihuahua
   1: Shih-Tzu
   2: Rhodesian_ridgeback
   3: beagle
   4: English_foxhound
   5: Italian_greyhound
   6: Weimaraner
   7: Staffordshire_bullterrier
   8: Yorkshire_terrier
   9: Boston_bull
  10: miniature_schnauzer
  11: golden_retriever
  12: Labrador_retriever
  13: Irish_setter
  14: cocker_spaniel
  15: Shetland_sheepdog
  16: Border_collie
  17: Bouvier_des_Flandres
  18: Rottweiler
  19: German_shepherd
  20: Doberman
  21: Bernese_mountain_dog
  22: boxer
  23: French_bulldog
  24: Great_Dane
  25: Saint_Bernard
  26: Siberian_husky
  27: pug
  28: Samoyed
  29: Pomeranian


3. extracts the bounding box coordinate and normalize it between 0, 1. 0 is the left/top of the image and the 1 is the right/bottom.

In [ ]:
image_paths = []
labels = []
bboxes = []
skipped = 0

for breed_folder in SELECTED_BREEDS:
    label = breed_to_label[breed_folder]
    img_dir = IMAGES_DIR / breed_folder
    ann_dir = ANNOTATIONS_DIR / breed_folder

    for img_file in sorted(img_dir.glob("*.jpg")):
        ann_file = ann_dir / img_file.stem
        if not ann_file.exists():
            skipped += 1
            continue
        try:
            bbox = parse_annotation(ann_file)
            image_paths.append(str(img_file))
            labels.append(label)
            bboxes.append(bbox)
        except Exception:
            skipped += 1

image_paths = np.array(image_paths)
labels = np.array(labels)
bboxes = np.array(bboxes, dtype=np.float32)

print(f"Total samples: {len(image_paths)} (skipped {skipped})")
print(f"Labels shape: {labels.shape}, Bboxes shape: {bboxes.shape}")
print(f"\nSamples per breed:")
counts = Counter(labels)
for lbl in sorted(counts):
    print(f"  {label_to_breed[lbl]:30s}: {counts[lbl]}")

In [ ]:
# Stratified split: 70% train, 15% validation, 15% test
# stratify=labels ensures each breed has the same proportion in every split

X_train, X_temp, y_train, y_temp, bbox_train, bbox_temp = train_test_split(
    image_paths, labels, bboxes,
    test_size=0.30, stratify=labels, random_state=SEED
)

X_val, X_test, y_val, y_test, bbox_val, bbox_test = train_test_split(
    X_temp, y_temp, bbox_temp,
    test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f"Train:      {len(X_train)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Test:       {len(X_test)} samples")

print(f"\nBreed distribution check (first 5 breeds):")
for lbl in range(5):
    name = label_to_breed[lbl]
    tr = np.sum(y_train == lbl)
    va = np.sum(y_val == lbl)
    te = np.sum(y_test == lbl)
    print(f"  {name:25s} -> train={tr}, val={va}, test={te}")

In [ ]:
train_ds = build_dataset(X_train, y_train, bbox_train, shuffle=True)
val_ds = build_dataset(X_val, y_val, bbox_val)
test_ds = build_dataset(X_test, y_test, bbox_test)

for images, targets in train_ds.take(1):
    print(f"Image batch shape:  {images.shape}")
    print(f"Labels batch shape: {targets['class_output'].shape}")
    print(f"Bbox batch shape:   {targets['bbox_output'].shape}")
    print(f"Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for images, targets in train_ds.take(1):
    for i in range(8):
        img = images[i].numpy()
        label = targets["class_output"][i].numpy()
        bbox = targets["bbox_output"][i].numpy()

        xmin, ymin, xmax, ymax = bbox
        h, w = IMG_SIZE, IMG_SIZE

        axes[i].imshow(img)
        rect = plt.Rectangle(
            (xmin * w, ymin * h), (xmax - xmin) * w, (ymax - ymin) * h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
        axes[i].add_patch(rect)
        axes[i].set_title(label_to_breed[label], fontsize=10)
        axes[i].axis("off")

plt.suptitle("Training samples with bounding boxes", fontsize=14)
plt.tight_layout()
plt.show()